In [5]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
import time
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts  import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

C:\Users\dell\AppData\Local\Temp\ipykernel_33784\892504713.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [7]:
load_dotenv()

GEMINI_API= os.getenv("GEMINI_API_KEY")


In [8]:
loader= PyPDFLoader(r"C:\projects\acko_insurance_ai_platform\data\raw\Acko_Insurance_FAQs.pdf")
pages= loader.load()

In [9]:
print(len(pages))
print(pages[0].page_content[:500])

60
acko
Frequently Asked Questions
Motor | Health | Claims | Renewals | Legal | 70+ Questions
Page 1
AI-Generated Content  |  For Educational & RAG Project Use Only
acko.com  |  Welcome Change
acko
Frequently Asked Questions
Motor Insurance | Health Insurance | Claims | Renewals | Add-ons | Premiums
70+ Questions Answered in Detail — Complete Customer Reference Guide
 Motor
 Health
 Claims
 NCB
 Add-ons
 Renewals
Document Type
Frequently Asked Questions — Customer Reference Guide
Topics Covered
Car


In [10]:
print(pages[30].page_content[0:1000])

acko
Frequently Asked Questions
Supplementary Sections K-N
Page 5
AI-Generated Content  |  For Educational & RAG Project Use Only
acko.com  |  Welcome Change
SECTION M — INSURANCE PLANNING GUIDE FOR LIFE STAGES
Insurance needs change through life stages. Here is a life-stage planning guide for motor and health insurance:
Stage 1: Young Professional (22–28 Years)
First job, first vehicle, starting financial independence.
Motor Insurance
Buy Comprehensive with Zero Dep for new car/bike. Start NCB accumulation — make
no minor claims.
Health Insurance
Buy personal health policy (INR 5-10 lakh) immediately — even if employer covers you.
Start waiting period clock.
Add-ons
Zero Dep (motor), RSA (motor). No add-on health immediately if budget constrained.
Priority
Start NCB. Serve PED waiting periods. Build habit of renewing on time.
Stage 2: Married Couple (28–35 Years)
Dual income, possibly first home loan, planning for family.
Motor Insurance
Both vehicles on Comprehensive. NCB building on

In [11]:
text_splitter= RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap= 100)
texts= text_splitter.split_documents(pages)

In [12]:
print(len(texts))

239


In [13]:
print(texts[0].page_content)
print("---")
print(texts[1].page_content)
print("---")
print(texts[50].page_content)

acko
Frequently Asked Questions
Motor | Health | Claims | Renewals | Legal | 70+ Questions
Page 1
AI-Generated Content  |  For Educational & RAG Project Use Only
acko.com  |  Welcome Change
acko
Frequently Asked Questions
Motor Insurance | Health Insurance | Claims | Renewals | Add-ons | Premiums
70+ Questions Answered in Detail — Complete Customer Reference Guide
 Motor
 Health
 Claims
 NCB
 Add-ons
 Renewals
Document Type
Frequently Asked Questions — Customer Reference Guide
Topics Covered
Car Insurance, Bike/EV Insurance, Health Insurance, Claims, NCB, Add-ons, Renewals,
Digital Services, Premiums, Portability
Total Questions
70+ FAQs across 10 sections
Content Status
AI-Generated | Educational & RAG Project Use Only
Purpose
---
Content Status
AI-Generated | Educational & RAG Project Use Only
Purpose
RAG Model Knowledge Base | Student Reference | Educational Training
Language
English | Jurisdiction: Republic of India
NOTICE: This FAQ document is AI-generated for educational purposes

In [14]:
print(texts[0].metadata)

{'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'C:\\projects\\acko_insurance_ai_platform\\data\\raw\\Acko_Insurance_FAQs.pdf', 'total_pages': 60, 'page': 0, 'page_label': '1'}


In [15]:
print(texts[50].metadata)

{'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'C:\\projects\\acko_insurance_ai_platform\\data\\raw\\Acko_Insurance_FAQs.pdf', 'total_pages': 60, 'page': 11, 'page_label': '12'}


### Loading all pdf's

In [16]:
pdf_folder=  (r"C:\projects\acko_insurance_ai_platform\data\raw")
all_pages= []

for filename in os.listdir(pdf_folder):
    if filename.endswith('.pdf'):
        loader= PyPDFLoader(os.path.join(pdf_folder, filename))
        pages= loader.load()
        all_pages.extend(pages)


In [17]:
print(len(all_pages))

167


In [18]:
# text splitting

In [19]:
text_splitter= RecursiveCharacterTextSplitter(chunk_size= 800, chunk_overlap=100)
chunks= text_splitter.split_documents(all_pages)

In [20]:
len(chunks)

595

In [21]:
# Embedding the chunks

In [22]:
embeddings= GoogleGenerativeAIEmbeddings(model= "gemini-embedding-2-preview", 
                                google_api_key=os.getenv("GEMINI_API_KEY"))

In [23]:
#vector Store

In [24]:
batch_size= 50
vectorstore= None

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]
    print(f"Embedding batch {i // batch_size + 1} of {(len(chunks) - 1) // batch_size + 1}...")

    if vectorstore is None:
        vectorstore= Chroma.from_documents(documents= batch, embedding= embeddings,
                        persist_directory=r"C:\projects\acko_insurance_ai_platform\data\processed\chroma_db")
    else:
        vectorstore.add_documents(batch)
    
    time.sleep(60)
print("Done embedding all batches.")


Embedding batch 1 of 12...
Embedding batch 2 of 12...
Embedding batch 3 of 12...
Embedding batch 4 of 12...
Embedding batch 5 of 12...
Embedding batch 6 of 12...
Embedding batch 7 of 12...
Embedding batch 8 of 12...
Embedding batch 9 of 12...
Embedding batch 10 of 12...
Embedding batch 11 of 12...
Embedding batch 12 of 12...
Done embedding all batches.


In [25]:
results = vectorstore.similarity_search("What is NCB and how does it work?", k=3)
for r in results:
    print(r.page_content[:300])
    print("---")

Compulsory Personal Accident (CPA) claims = NCB unaffected. The only thing that affects NCB: Making an OD claim
where your own vehicle is repaired or replaced using your own insurance. Even a INR 500 OD claim resets NCB to 0%
unless NCB Protection add-on is active.
Q  E3. I just made a claim and los
---
  NCB applies ONLY to the Own Damage premium — not to Third Party premium or Personal Accident cover.
 
  Even one OD claim in a policy year resets NCB to 0% at renewal — unless NCB Protection Add-on is active.
 
  Third Party claims and Personal Accident claims do NOT affect NCB.
 
  NCB is for
---
NCB Protection add-on before making any future claim — protects the NCB you rebuild. If you switch insurers: Request an
NCB certificate from Acko (issued in 3 working days; valid 90 days) and use it with the new insurer — your NCB is portable.
Q  E4. How do I transfer my NCB when I buy a new car?
A 
---


In [26]:
results2 = vectorstore.similarity_search("What is covered under health insurance policy?", k=3)
for r in results2:
    print(r.page_content[:300])
    print("---")

Per IRDAI schedule — not provider tariff
Anaesthetist's Fees
In full
Standard rates apply
Specialist Consultation
In full during hospitalisation
Pre-authorised specialists
Operation Theatre Charges
In full including disposables
Standard OT rates apply
Medicines and Drugs
As prescribed for in-patient
---
Pre-Hospitalisation
60 days before admission
Post-Hospitalisation
90 days after discharge
Daycare Procedures
541 listed procedures covered
Ambulance
INR 5,000 per hospitalisation
AYUSH Treatment
In-patient only — up to 25% of Sum Insured
Mental Health Cover
Full coverage per IRDAI 2023 guidelines
Re
---
Stents, pacemakers, prosthetics
Hospital Nursing Charges
Included in room rate or billed separately
Both covered up to Room Rent Limit
4.2 Pre-Hospitalisation Expenses — 60 Days
Medical expenses incurred for the SAME illness or injury for which subsequent hospitalisation occurs, up to 60 days before
---


In [24]:
#Retriever

In [27]:
llm = ChatGoogleGenerativeAI(model= "gemini-3.5-flash", temperature= 0.2, google_api_key= os.getenv("GEMINI_API_KEY"))

In [28]:
SYSTEM_PROMPT = """
You are Acko's insurance assistant, helping customers understand their motor and health insurance policies along with FAQ's.

Guidelines:
- Answer ONLY using the context provided below. Do not use outside knowledge.
- If the answer isn't in the provided context, say clearly: "I don't have that information in the available policy documents. Please contact Acko support for details."
- Never invent specific numbers, coverage limits, or policy terms that aren't explicitly stated in the context.
- Be concise and clear — prefer short, direct answers over long explanations.
- Be polite and professional. Greet the customer warmly if this is the start of a conversation.
- If relevant, mention which policy type (Motor/Health) or section the answer comes from.
- This chatbot does not provide legal or financial advice — for complex claims or disputes, direct the customer to Acko's official support channels.

Context:
{context}

Question:
{question}

Answer:
"""

In [29]:
#prompt

In [30]:
prompt= ChatPromptTemplate.from_template(SYSTEM_PROMPT)

In [31]:
retriever= vectorstore.as_retriever(search_kwargs= {"k": 3})

In [34]:
#Combining prompt+ LLM

In [35]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [36]:
rag_chain= ({"context": retriever | format_docs , "question" : RunnablePassthrough()}
            |prompt
            | llm
            |StrOutputParser())

In [37]:
answer= rag_chain.invoke("What is NCB and how does it work?")
print(answer)

Hello! I would be happy to help you understand how NCB works under your Motor insurance policy. 

Based on the provided policy documents, here is how NCB works:

* **What it applies to:** NCB applies ONLY to the Own Damage (OD) premium. It does not apply to Third Party premiums or Personal Accident cover.
* **How claims affect it:** Making even a single Own Damage (OD) claim (even as low as INR 500) resets your NCB to 0% at the next renewal, unless you have the NCB Protection Add-on active. Third Party claims and Compulsory Personal Accident (CPA) claims do not affect your NCB.
* **Rebuilding NCB:** If your NCB resets to 0% after a claim, you can rebuild it by going consecutive years without making an OD claim:
  * **Year 1:** 20% NCB
  * **Year 2:** 25% NCB
  * **Year 3:** 35% NCB
  * **Year 4:** 45% NCB
  * **Year 5:** 50% NCB
* **Transfer and Portability:** NCB is linked to you (the policyholder), not your vehicle. You can transfer it to a different make or model of vehicle, or port

In [38]:
answer2 = rag_chain.invoke("What's the weather like today in Chennai?")
print(answer2)

Hello! I don't have that information in the available policy documents. Please contact Acko support for details.


In [39]:
answer3 = rag_chain.invoke("Does Acko offer travel insurance for international trips?")
print(answer3)

Hello! I don't have that information in the available policy documents. Please contact Acko support for details.
